In [0]:
%sql SELECT COUNT(*) FROM real_state_project_bundle.gold.fact_inmuebles_mercado

In [0]:
import json
from pyspark.sql.functions import current_timestamp, input_file_name, lit, regexp_extract,col

# ==============================================================================
# PASO 1: DECLARAR LOS WIDGETS (Solo el Catálogo es dinámico para el entorno)
# ==============================================================================
#dbutils.widgets.text("catalog_name", "") 

# Parámetros inyectados por el For Each de tu tabla de control
dbutils.widgets.text("scraper_tool", "")
dbutils.widgets.text("portal", "")          
dbutils.widgets.text("property_type", "")
dbutils.widgets.text("operation_type", "")
dbutils.widgets.text("file_prefix", "")
dbutils.widgets.text("target_table", "")     # Ej: "bronze.inmuebles24_raw"
dbutils.widgets.text("read_options", "{}")

# Capturar valores
#catalog_name = dbutils.widgets.get("catalog_name")
catalog_name = "real_state_project"
scraper_tool = dbutils.widgets.get("scraper_tool")
portal = dbutils.widgets.get("portal")      
property_type = dbutils.widgets.get("property_type")
operation_type = dbutils.widgets.get("operation_type")
file_prefix = dbutils.widgets.get("file_prefix")
schema_and_table = dbutils.widgets.get("target_table")
read_options_str = dbutils.widgets.get("read_options")

if not catalog_name or not file_prefix:
    dbutils.notebook.exit("Error: Faltan parámetros críticos para la ingesta.")

# ==============================================================================
# PASO 2: CONSTRUCCIÓN DINÁMICA DE RUTAS BASADAS EN VOLUMES
# ==============================================================================
read_options_dict = json.loads(read_options_str)
full_target_table = f"{catalog_name}.{schema_and_table}"

# La ruta del Volume se adapta al catálogo del entorno de forma automática
source_path = f"/Volumes/{catalog_name}/raw_data_real_state/real_state_csv_landing_volume/{file_prefix}*"

# Creamos una carpeta de checkpoints dedicada dentro de un Volume para mantener la gobernanza
checkpoint_path = f"/Volumes/{catalog_name}/raw_data_real_state/real_state_csv_landing_volume/_checkpoints/{file_prefix}"
schema_location_path = f"/Volumes/{catalog_name}/raw_data_real_state/real_state_csv_landing_volume/_schemas/{file_prefix}"
# ==============================================================================
# PASO 3: LECTURA INCREMENTAL CON AUTO LOADER
# ==============================================================================
print(f"Leyendo desde Volume: {source_path}")
print(f"Destino en Unity Catalog: {full_target_table}")
spark.catalog.setCurrentCatalog(catalog_name)
df_raw = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
    .option("cloudFiles.schemaLocation", schema_location_path) \
    .options(**read_options_dict) \
    .load(source_path)

# ==============================================================================
# PASO 4: ENRIQUECIMIENTO Y METADATOS
# ==============================================================================
df_enriched = df_raw \
    .withColumn("scraper_tool", lit(scraper_tool)) \
    .withColumn("portal", lit(portal)) \
    .withColumn("property_type", lit(property_type)) \
    .withColumn("operation_type", lit(operation_type)) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("extraction_date", regexp_extract("source_file", r"(\d{4}-\d{2}-\d{2})", 1)) \
    .withColumn("ingested_at", current_timestamp())

# ==============================================================================
# PASO 5: ESCRITURA EN CAPA BRONZE
# ==============================================================================
query = df_enriched.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(schema_and_table)

query.awaitTermination()
dbutils.notebook.exit(f"Éxito: Ingesta completada en {full_target_table}")